# 作业 3.1：HPGe $\gamma$ 能谱刻度

## 实验装置、标准源与 ROOT 文件

### 实验装置

Eurica 探测阵列由 12 个 Euroball Cluster 组成，每个 Cluster 含 7 个 HPGe 晶体。标定源距探测器约 22 cm。数据已对每个 Cluster 的 7 个晶体进行 add-back，再合并 12 个 Cluster。

<img src="eurica.png" alt="Eurica detector array" style="max-width:50%;" />

### 标准源

能谱由 $^{152}$Eu 与 $^{133}$Ba 标准源测得，数据采集于 2013 年 2 月 13 日。源的参考日期为 1998 年 1 月 1 日；参考活度为 $^{152}$Eu 40.9 kBq（5%）和 $^{133}$Ba 42.2 kBq（3%）。活度衰变修正可采用 $T_{1/2}(^{152}\mathrm{Eu})=13.517$ y、$T_{1/2}(^{133}\mathrm{Ba})=3849.3$ d。

$P_\gamma$ 是每次核衰变发出该 gamma ray 的概率，不是源活度。

| Nuclide | $E_\gamma$ (keV) | $P_\gamma$ (%) |
| --- | ---: | ---: |
| $^{133}$Ba | 80.9979 | 34.06 |
| $^{152}$Eu | 121.7817 | 28.41 |
| $^{152}$Eu | 244.6974 | 7.55 |
| $^{133}$Ba | 276.3989 | 7.164 |
| $^{133}$Ba | 302.8508 | 18.33 |
| $^{152}$Eu | 344.2785 | 26.59 |
| $^{133}$Ba | 356.0129 | 62.05 |
| $^{152}$Eu | 778.9045 | 12.93 |
| $^{152}$Eu | 867.378 | 4.23 |
| $^{152}$Eu | 964.079 | 14.51 |
| $^{152}$Eu | 1112.076 | 13.67 |
| $^{152}$Eu | 1408.013 | 20.87 |

### ROOT 文件

使用 [gamma.root](gamma.root) 中的 `TH1F h0`。`h0` 的横轴是尚未刻度的 channel，范围为 0–2500，每个 bin 宽 0.2 channel。记录时长为 7442 s。效率计算中先把 7442 s 当作 live time；若它实际是 wall-clock time，则还需要 dead-time correction。

## 方法

能量与 channel 近似满足线性关系。先在 log scale 下寻找较显著的峰候选；对同一核素，可用 $P_\gamma$ 的相对大小辅助判断，对两个不同核素则还要计入各自的 $A(t)$。选择两个相隔较远且指认可靠的峰估计初步线性关系，再用它预测并核对其他刻度线的位置。观测峰强还随 full-energy peak efficiency 改变，因此 $A(t)P_\gamma$ 只用于初步定位，不能直接当作峰面积之比。

HPGe 能谱刻度从局部峰拟合开始。对每个刻度峰采用同一个

$$f(x)=s(x)+b(x)$$

模型：$s(x)$ 是 photopeak signal，$b(x)$ 是局部本底。孤立且近似对称的峰可用 Gaussian signal；本底先用线性函数，低能侧形成明显 step 时再加入 `erfc` 项。模型是否足够由拟合残差判断。

<img src="../calibration_method/peak_model_components.png" alt="photopeak signal, local background, total fit and residual" style="max-width:55%;" />

同一次拟合给出三类刻度量：

- 峰中心 $ch_i$ 用于能量刻度。先拟合 $E=a_0+a_1ch$，只有残差显示系统曲率时才加入 $a_2ch^2$。
- $\sigma_{ch}$ 先换算为 $\sigma_E=|dE/dch|\sigma_{ch}$，再计算 $FWHM=2.355\sigma_E$。
- 峰面积取信号分量的积分。对等宽 histogram 中的 Gaussian peak，

  $$N_{\mathrm{peak}}=\frac{A\sigma\sqrt{2\pi}}{w},$$

  其中 $A$ 是 Gaussian height，$w$ 是 bin width。这里不把 covariance 作为峰拟合的输入；峰拟合完成后，用 ROOT 返回的参数 covariance matrix 中的 $\operatorname{Cov}(A,\sigma)$ 传播峰面积误差。

<img src="../calibration_method/full_energy_peak_area.png" alt="full-energy peak area above the fitted local background" style="max-width:55%;" />

Full-energy peak efficiency 使用同一个 $N_{\mathrm{peak}}$：

$$
\varepsilon(E_\gamma)=
\frac{N_{\mathrm{peak}}}{A(t)P_\gamma t_{\mathrm{live}}}.
$$

$A(t)$ 是测量时活度，$P_\gamma$ 是每次衰变发射该 gamma ray 的概率。Sideband subtraction 只作为独立交叉检查，不与上述拟合本底混用。

## 作业要求

### 能量刻度

1. 用 log scale 查看完整的 `h0`，按照线性关系和相对发射强度初步定位刻度峰。
2. 对各刻度峰进行局部拟合。峰与邻近结构重叠时应调整拟合区间，不能让本底函数吸收另一个峰。逐峰检查拟合残差。
3. 将峰中心和参考能量写入 `TGraphErrors`，峰中心误差作为横坐标误差。比较一次和二次刻度函数，并画出 $\Delta E=E_{\mathrm{ref}}-E_{\mathrm{cal}}$ 残差。
4. 用残差支持的刻度关系生成能量谱，并检查变换前后总计数是否守恒。

### 峰宽与能量分辨率

由各峰的 $\sigma_{ch}$ 计算 FWHM，画出 FWHM–$E_\gamma$ 曲线，拟合

$$FWHM(E)=\sqrt{A+BE+CE^2},$$

并给出 $FWHM_{\mathrm{data}}-FWHM_{\mathrm{fit}}$ 残差。根据残差判断是否需要全部三项。

### Full-energy peak efficiency（选做）

1. 从峰形拟合中的 signal 分量计算 $N_{\mathrm{peak}}$，并用拟合返回的参数 covariance matrix 传播其统计误差。
2. 将两个源的活度修正到测量日期，计算各条谱线的 apparent full-energy peak efficiency。
3. 在 log–log 坐标上拟合 efficiency–energy 曲线，并给出相对残差。
4. 说明 dead time、true-coincidence summing、源几何和自吸收修正对绝对效率的影响。

## 实例代码：两个典型峰

下面按照一次实际局部拟合的顺序展开代码。第一个峰使用 Gaussian signal + linear background；第二个峰说明低能侧存在 step 时如何修改本底。两个例子都给出峰中心、$\sigma$、signal 面积及其误差，并检查拟合残差。其余刻度峰仍需自行定位和拟合。

<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight {
  margin:.5rem 0 1rem;
  border:1px solid #d5d5d5;
  border-radius:2px;
  background:#f7f7f7;
}
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden],
.jp-CodeCell .jp-Cell-inputWrapper[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>

### 读取并查看能谱

先打开 `gamma.root`，取得 `h0`，用 log scale 查看完整能谱。这里只改变显示范围，不改变 histogram 中的计数。

<div class="pyroot-code-marker"></div>

```python
import math
import ROOT

ROOT.gStyle.SetOptStat(0)

# TFile.Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
input_file = ROOT.TFile.Open("gamma.root", "READ")
h0 = input_file.Get("h0")

c_spectrum = ROOT.TCanvas("c_spectrum_py", "h0", 850, 480)
c_spectrum.SetLogy()
h0.SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin")
h0.GetXaxis().SetRangeUser(40, 1300)
h0.SetMinimum(0.5)
h0.Draw("hist")
c_spectrum.Draw()
```

In [1]:
#include "TCanvas.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TH1.h"
#include "TLine.h"
#include "TMath.h"
#include "TStyle.h"
#include <algorithm>
#include <cmath>
#include <iostream>

gStyle->SetOptStat(0);

// TFile::Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
auto inputFile = TFile::Open("gamma.root", "READ");
auto h0 = dynamic_cast<TH1*>(inputFile->Get("h0"));

auto cSpectrum = new TCanvas("cSpectrum", "h0", 850, 480);
cSpectrum->SetLogy();
h0->SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin");
h0->GetXaxis()->SetRangeUser(40, 1300);
h0->SetMinimum(0.5);
h0->Draw("hist");
cSpectrum->Draw();

### Gaussian signal + linear background

先选定一个孤立峰的局部区间，并从谱图估计峰高、峰中心、$\sigma$ 和本底的初值。参数 0–2 属于 Gaussian；线性本底写成 `[3]+[4]*(x-239.1)`，使参数 3 直接表示峰附近的本底高度。`Fit` 的 `LIRSQN` 选项依次表示 binned Poisson likelihood、对每个 bin 积分、使用函数区间、返回拟合结果、静默且不自动画图。

<div class="pyroot-code-marker"></div>

```python
xmin244, xmax244 = 235.6, 242.6
f244 = ROOT.TF1("f244_py", "gaus(0)+[3]+[4]*(x-239.1)", xmin244, xmax244)
f244.SetParNames("height", "mean", "sigma", "b0", "b1")
f244.SetParameters(1.8e5, 239.1, 0.7, 4.0e4, 0.0)
f244.SetParLimits(0, 0.0, 1.0e7)
f244.SetParLimits(1, 237.0, 241.0)
f244.SetParLimits(2, 0.2, 3.0)

result244 = h0.Fit(f244, "LIRSQN")

# GetParameter / GetParError 读取拟合参数及其标准误差。
height244 = f244.GetParameter(0)
mean244 = f244.GetParameter(1)
sigma244 = abs(f244.GetParameter(2))

# Gaussian height 和 sigma 共同决定峰面积，面积误差需要二者的 covariance。
area_factor = math.sqrt(2.0 * math.pi) / h0.GetBinWidth(1)
area244 = height244 * sigma244 * area_factor
cov244 = result244.GetCovarianceMatrix()
area_variance244 = (
    (sigma244 * area_factor)**2 * cov244[0][0]
    + (height244 * area_factor)**2 * cov244[2][2]
    + 2.0 * height244 * sigma244 * area_factor**2 * cov244[0][2]
)
area_error244 = math.sqrt(max(area_variance244, 0.0))

print("244.7 keV example")
print(f"mean   = {mean244:.4f} +/- {f244.GetParError(1):.4f}")
print(f"sigma  = {sigma244:.4f}")
print(f"N_peak = {area244:.0f} +/- {area_error244:.0f}")
print(f"fit status = {int(result244)}")
```

In [2]:
double xmin244 = 235.6;
double xmax244 = 242.6;
auto f244 = new TF1("f244", "gaus(0)+[3]+[4]*(x-239.1)", xmin244, xmax244);
f244->SetParNames("height", "mean", "sigma", "b0", "b1");
f244->SetParameters(1.8e5, 239.1, 0.7, 4.0e4, 0.0);
f244->SetParLimits(0, 0.0, 1.0e7);
f244->SetParLimits(1, 237.0, 241.0);
f244->SetParLimits(2, 0.2, 3.0);

// L/I/R/S/Q/N: Poisson likelihood / bin integral / function range /
// return result / quiet / do not draw automatically.
TFitResultPtr result244 = h0->Fit(f244, "LIRSQN");

// GetParameter / GetParError 读取拟合参数及其标准误差。
double height244 = f244->GetParameter(0);
double mean244 = f244->GetParameter(1);
double sigma244 = std::abs(f244->GetParameter(2));

// Gaussian height 和 sigma 共同决定峰面积。
double areaFactor = std::sqrt(2.0 * TMath::Pi()) / h0->GetBinWidth(1);
double area244 = height244 * sigma244 * areaFactor;
auto cov244 = result244->GetCovarianceMatrix();
double areaVariance244 =
    std::pow(sigma244 * areaFactor, 2) * cov244(0, 0)
    + std::pow(height244 * areaFactor, 2) * cov244(2, 2)
    + 2.0 * height244 * sigma244 * areaFactor * areaFactor * cov244(0, 2);
double areaError244 = std::sqrt(std::max(areaVariance244, 0.0));

std::cout << "244.7 keV example\n"
          << "mean   = " << mean244 << " +/- " << f244->GetParError(1) << "\n"
          << "sigma  = " << sigma244 << "\n"
          << "N_peak = " << area244 << " +/- " << areaError244 << "\n"
          << "fit status = " << static_cast<int>(result244) << "\n";

244.7 keV example
mean   = 239.191 +/- 0.000795361
sigma  = 0.651885
N_peak = 1.42987e+06 +/- 1758.13
fit status = 0


拟合曲线接近数据并不等于模型已经充分。下面逐 bin 计算 Pearson residual，$r_i=(n_i-\mu_i)/\sqrt{\mu_i}$；其中 $\mu_i$ 使用拟合函数在该 bin 内的平均值，与 `I` 选项一致。

<div class="pyroot-code-marker"></div>

```python
residual244 = ROOT.TGraph()
for point, bin_number in enumerate(
    range(h0.FindBin(xmin244), h0.FindBin(xmax244) + 1)
):
    low = h0.GetBinLowEdge(bin_number)
    width = h0.GetBinWidth(bin_number)
    expected = f244.Integral(low, low + width) / width
    observed = h0.GetBinContent(bin_number)
    residual244.SetPoint(
        point, h0.GetBinCenter(bin_number),
        (observed - expected) / math.sqrt(expected)
    )

c244 = ROOT.TCanvas("c244_py", "244.7 keV example", 720, 650)
c244.Divide(1, 2)
c244.cd(1)
h244_view = h0.Clone("h244_view_py")
h244_view.GetXaxis().SetRangeUser(xmin244, xmax244)
h244_view.SetTitle("244.7 keV example;channel;counts / bin")
h244_view.Draw("E")
f244.SetLineColor(ROOT.kBlue + 1)
f244.Draw("same")
c244.cd(2)
residual244.SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}")
residual244.SetMarkerStyle(20)
residual244.Draw("AP")
zero244 = ROOT.TLine(xmin244, 0.0, xmax244, 0.0)
zero244.SetLineStyle(2)
zero244.Draw()
c244.Draw()
```

In [3]:
auto residual244 = new TGraph();
int point244 = 0;
for (int bin = h0->FindBin(xmin244); bin <= h0->FindBin(xmax244); ++bin) {
    double low = h0->GetBinLowEdge(bin);
    double width = h0->GetBinWidth(bin);
    double expected = f244->Integral(low, low + width) / width;
    double observed = h0->GetBinContent(bin);
    residual244->SetPoint(point244++, h0->GetBinCenter(bin),
                          (observed - expected) / std::sqrt(expected));
}

auto c244 = new TCanvas("c244", "244.7 keV example", 720, 650);
c244->Divide(1, 2);
c244->cd(1);
auto h244View = static_cast<TH1*>(h0->Clone("h244View"));
h244View->GetXaxis()->SetRangeUser(xmin244, xmax244);
h244View->SetTitle("244.7 keV example;channel;counts / bin");
h244View->Draw("E");
f244->SetLineColor(kBlue + 1);
f244->Draw("same");
c244->cd(2);
residual244->SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}");
residual244->SetMarkerStyle(20);
residual244->Draw("AP");
auto zero244 = new TLine(xmin244, 0.0, xmax244, 0.0);
zero244->SetLineStyle(2);
zero244->Draw();
c244->Draw();

### 在局部本底中加入 step

第二个例子的低能侧本底明显较高，因此在 Gaussian + linear background 上加入与峰中心和峰宽相连的 `erfc` step。下面仍按“设定区间和初值 → 拟合 → 提取参数与面积”的顺序进行。

<div class="pyroot-code-marker"></div>

```python
# 上一个例子改变过显示范围；再次拟合前先恢复完整横轴范围。
h0.GetXaxis().SetRangeUser(0.0, 2500.0)
xmin867, xmax867 = 758.2, 767.2
model867 = (
    "gaus(0)+[3]+[4]*(x-762.7)"
    "+[5]*0.5*TMath::Erfc((x-[1])/(sqrt(2)*[2]))"
)
f867 = ROOT.TF1("f867_py", model867, xmin867, xmax867)
f867.SetParNames("height", "mean", "sigma", "b0", "b1", "step")
f867.SetParameters(4.5e4, 762.7, 0.8, 6.0e3, 0.0, 1.5e3)
f867.SetParLimits(0, 0.0, 1.0e7)
f867.SetParLimits(1, 760.0, 765.0)
f867.SetParLimits(2, 0.2, 3.0)
f867.SetParLimits(5, 0.0, 1.0e6)

result867 = h0.Fit(f867, "LIRSQN")
height867 = f867.GetParameter(0)
mean867 = f867.GetParameter(1)
sigma867 = abs(f867.GetParameter(2))
area867 = height867 * sigma867 * area_factor
cov867 = result867.GetCovarianceMatrix()
area_variance867 = (
    (sigma867 * area_factor)**2 * cov867[0][0]
    + (height867 * area_factor)**2 * cov867[2][2]
    + 2.0 * height867 * sigma867 * area_factor**2 * cov867[0][2]
)
area_error867 = math.sqrt(max(area_variance867, 0.0))

print("867.4 keV example")
print(f"mean   = {mean867:.4f} +/- {f867.GetParError(1):.4f}")
print(f"sigma  = {sigma867:.4f}")
print(f"N_peak = {area867:.0f} +/- {area_error867:.0f}")
print(f"fit status = {int(result867)}")
```

In [4]:
// 上一个例子改变过显示范围；再次拟合前先恢复完整横轴范围。
h0->GetXaxis()->SetRangeUser(0.0, 2500.0);
double xmin867 = 758.2;
double xmax867 = 767.2;
auto f867 = new TF1(
    "f867",
    "gaus(0)+[3]+[4]*(x-762.7)+[5]*0.5*TMath::Erfc((x-[1])/(sqrt(2)*[2]))",
    xmin867, xmax867);
f867->SetParNames("height", "mean", "sigma", "b0", "b1", "step");
f867->SetParameters(4.5e4, 762.7, 0.8, 6.0e3, 0.0, 1.5e3);
f867->SetParLimits(0, 0.0, 1.0e7);
f867->SetParLimits(1, 760.0, 765.0);
f867->SetParLimits(2, 0.2, 3.0);
f867->SetParLimits(5, 0.0, 1.0e6);

TFitResultPtr result867 = h0->Fit(f867, "LIRSQN");
double height867 = f867->GetParameter(0);
double mean867 = f867->GetParameter(1);
double sigma867 = std::abs(f867->GetParameter(2));
double area867 = height867 * sigma867 * areaFactor;
auto cov867 = result867->GetCovarianceMatrix();
double areaVariance867 =
    std::pow(sigma867 * areaFactor, 2) * cov867(0, 0)
    + std::pow(height867 * areaFactor, 2) * cov867(2, 2)
    + 2.0 * height867 * sigma867 * areaFactor * areaFactor * cov867(0, 2);
double areaError867 = std::sqrt(std::max(areaVariance867, 0.0));

std::cout << "867.4 keV example\n"
          << "mean   = " << mean867 << " +/- " << f867->GetParError(1) << "\n"
          << "sigma  = " << sigma867 << "\n"
          << "N_peak = " << area867 << " +/- " << areaError867 << "\n"
          << "fit status = " << static_cast<int>(result867) << "\n";

867.4 keV example
mean   = 762.635 +/- 0.00222076
sigma  = 0.819858
N_peak = 480499 +/- 883.621
fit status = 0


最后用同样的方法计算第二个峰的 residual，并把数据、总拟合函数和 residual 放在同一张图中。

<div class="pyroot-code-marker"></div>

```python
residual867 = ROOT.TGraph()
for point, bin_number in enumerate(
    range(h0.FindBin(xmin867), h0.FindBin(xmax867) + 1)
):
    low = h0.GetBinLowEdge(bin_number)
    width = h0.GetBinWidth(bin_number)
    expected = f867.Integral(low, low + width) / width
    observed = h0.GetBinContent(bin_number)
    residual867.SetPoint(
        point, h0.GetBinCenter(bin_number),
        (observed - expected) / math.sqrt(expected)
    )

c867 = ROOT.TCanvas("c867_py", "867.4 keV example", 720, 650)
c867.Divide(1, 2)
c867.cd(1)
h867_view = h0.Clone("h867_view_py")
h867_view.GetXaxis().SetRangeUser(xmin867, xmax867)
h867_view.SetTitle("867.4 keV example;channel;counts / bin")
h867_view.Draw("E")
f867.SetLineColor(ROOT.kBlue + 1)
f867.Draw("same")
c867.cd(2)
residual867.SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}")
residual867.SetMarkerStyle(20)
residual867.Draw("AP")
zero867 = ROOT.TLine(xmin867, 0.0, xmax867, 0.0)
zero867.SetLineStyle(2)
zero867.Draw()
c867.Draw()
```

In [5]:
auto residual867 = new TGraph();
int point867 = 0;
for (int bin = h0->FindBin(xmin867); bin <= h0->FindBin(xmax867); ++bin) {
    double low = h0->GetBinLowEdge(bin);
    double width = h0->GetBinWidth(bin);
    double expected = f867->Integral(low, low + width) / width;
    double observed = h0->GetBinContent(bin);
    residual867->SetPoint(point867++, h0->GetBinCenter(bin),
                          (observed - expected) / std::sqrt(expected));
}

auto c867 = new TCanvas("c867", "867.4 keV example", 720, 650);
c867->Divide(1, 2);
c867->cd(1);
auto h867View = static_cast<TH1*>(h0->Clone("h867View"));
h867View->GetXaxis()->SetRangeUser(xmin867, xmax867);
h867View->SetTitle("867.4 keV example;channel;counts / bin");
h867View->Draw("E");
f867->SetLineColor(kBlue + 1);
f867->Draw("same");
c867->cd(2);
residual867->SetTitle("Fit residual;channel;(n-#mu)/#sqrt{#mu}");
residual867->SetMarkerStyle(20);
residual867->Draw("AP");
auto zero867 = new TLine(xmin867, 0.0, xmax867, 0.0);
zero867->SetLineStyle(2);
zero867->Draw();
c867->Draw();

## 参考结果

完成作业后，可用下面的结果检查刻度关系、曲线形状和残差结构。一次能量刻度给出

$$E_\gamma\;(\mathrm{keV})\approx-39.943+1.189802\,ch,$$

最大绝对刻度残差约为 0.10 keV；二次项没有改善本数据的最大残差。

<img src="reference_calibrated_spectrum.png" alt="calibrated gamma spectrum" style="max-width:55%;" />

<img src="reference_energy_calibration.png" alt="energy calibration and residuals" style="max-width:55%;" />

<img src="reference_fwhm.png" alt="FWHM versus energy and residuals" style="max-width:55%;" />

<img src="reference_efficiency.png" alt="apparent full-energy peak efficiency and residuals" style="max-width:55%;" />